In [1]:
import os
import glob
import cv2
import numpy as np
from pyneon import Recording

# ——— CONFIG ———
DATA_DIR       = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
OUTPUT_VIDEO   = '/Users/biancaschmid/Desktop/heatmap_overlay.mp4'
GAUSSIAN_SIGMA = 50   # spread of each gaze point
CIRCLE_RADIUS  = 30   # radius (px) of each fixation dot
ALPHA_VIDEO    = 0.6  # weight for original frame
ALPHA_HEATMAP  = 0.4  # weight for heatmap overlay
# ——————————

# 1) Load recording & pull out the gaze DataFrame
rec     = Recording(DATA_DIR)
stream  = rec.sync_gaze_to_video()
gaze_df = stream.data

# Drop any samples with missing coords
gaze_df = gaze_df.dropna(subset=['gaze x [px]', 'gaze y [px]', 'frame_idx'])

# 2) Locate the scene video
videos = sorted(glob.glob(os.path.join(DATA_DIR, '*.mp4')))
if not videos:
    raise FileNotFoundError(f"No .mp4 found in {DATA_DIR}")
video_path = videos[0]

# 3) Open video with OpenCV
cap      = cv2.VideoCapture(video_path)
fps      = cap.get(cv2.CAP_PROP_FPS)
width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 4) Prepare writer & accumulator
fourcc        = cv2.VideoWriter_fourcc(*'mp4v')
out           = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))
heatmap_accum = np.zeros((height, width), dtype=np.float32)

# 5) Frame-by-frame loop
for frame_idx in range(n_frames):
    ret, frame = cap.read()
    if not ret:
        break

    # select gaze samples for this frame
    pts = gaze_df[gaze_df['frame_idx'] == frame_idx]

    # draw each gaze point
    for _, row in pts.iterrows():
        x = int(row['gaze x [px]'])
        y = int(row['gaze y [px]'])
        # only if within bounds
        if 0 <= x < width and 0 <= y < height:
            cv2.circle(heatmap_accum, (x, y), CIRCLE_RADIUS, color=1.0, thickness=-1)

    # smooth & normalize
    heatmap_blur  = cv2.GaussianBlur(heatmap_accum, (0,0), GAUSSIAN_SIGMA, GAUSSIAN_SIGMA)
    heatmap_norm  = cv2.normalize(heatmap_blur, None, 0, 255, cv2.NORM_MINMAX)
    heatmap_uint8 = heatmap_norm.astype(np.uint8)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    # overlay & write
    overlay = cv2.addWeighted(frame, ALPHA_VIDEO, heatmap_color, ALPHA_HEATMAP, 0)
    out.write(overlay)

# 6) Cleanup
cap.release()
out.release()
print(f"✅ Saved heatmap‐overlay video to: {OUTPUT_VIDEO}")


✅ Saved heatmap‐overlay video to: /Users/biancaschmid/Desktop/heatmap_overlay.mp4


#with heatmap that updates itself

In [2]:
#only small part of video
import os
import glob
import cv2
import numpy as np
from pyneon import Recording

# ——— CONFIG ———
DATA_DIR          = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
OUTPUT_VIDEO      = '/Users/biancaschmid/Desktop/heatmap_test.mp4'
GAUSSIAN_SIGMA    = 50     # spread of each gaze point
CIRCLE_RADIUS     = 30     # radius (px) of each fixation dot
ALPHA_VIDEO       = 0.6    # weight for original frame
ALPHA_HEATMAP     = 0.4    # weight for heatmap overlay
DECAY_RATE        = 0.95   # per-frame decay of old heat (0 < rate < 1)

# Only process a small test window for speed
TEST_START_FRAME  = 0      # first frame to process
TEST_FRAME_COUNT  = 200    # number of frames to run
# ——————————

# 1) Load the Neon recording & sync gaze
rec     = Recording(DATA_DIR)
stream  = rec.sync_gaze_to_video()
gaze_df = stream.data.dropna(subset=['gaze x [px]', 'gaze y [px]', 'frame_idx'])

# 2) Find the scene video file
videos = sorted(glob.glob(os.path.join(DATA_DIR, '*.mp4')))
if not videos:
    raise FileNotFoundError(f"No .mp4 files found in {DATA_DIR}")
video_path = videos[0]

# 3) Open video with OpenCV
cap           = cv2.VideoCapture(video_path)
fps           = cap.get(cv2.CAP_PROP_FPS)
width         = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 4) Compute test window
end_frame = min(TEST_START_FRAME + TEST_FRAME_COUNT, total_frames)
cap.set(cv2.CAP_PROP_POS_FRAMES, TEST_START_FRAME)

# 5) Prepare VideoWriter and heatmap accumulator
fourcc        = cv2.VideoWriter_fourcc(*'mp4v')
out           = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))
heatmap_accum = np.zeros((height, width), dtype=np.float32)

# 6) Loop over test frames
for idx in range(TEST_START_FRAME, end_frame):
    ret, frame = cap.read()
    if not ret:
        break

    # 6a) decay old heat
    heatmap_accum *= DECAY_RATE

    # 6b) add current-frame gaze points
    pts = gaze_df[gaze_df['frame_idx'] == idx]
    for _, row in pts.iterrows():
        x = int(row['gaze x [px]'])
        y = int(row['gaze y [px]'])
        if 0 <= x < width and 0 <= y < height:
            cv2.circle(heatmap_accum, (x, y), CIRCLE_RADIUS, color=1.0, thickness=-1)

    # 6c) smooth & normalize
    heatmap_blur  = cv2.GaussianBlur(
        heatmap_accum, (0, 0),
        sigmaX=GAUSSIAN_SIGMA, sigmaY=GAUSSIAN_SIGMA
    )
    heatmap_norm  = cv2.normalize(heatmap_blur, None, 0, 255, cv2.NORM_MINMAX)
    heatmap_uint8 = heatmap_norm.astype(np.uint8)

    # 6d) colorize & overlay
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    overlay       = cv2.addWeighted(frame, ALPHA_VIDEO, heatmap_color, ALPHA_HEATMAP, 0)

    # 6e) write out
    out.write(overlay)

# 7) Clean up
cap.release()
out.release()
print(f"✅ Test heatmap video saved to:\n   {OUTPUT_VIDEO}")


✅ Test heatmap video saved to:
   /Users/biancaschmid/Desktop/heatmap_test.mp4


The “forgetting” of old heat in our dynamic map is governed entirely by the `DECAY_RATE` parameter, which applies an exponential decay to the accumulator every frame:

```python
heatmap_accum *= DECAY_RATE
```

– **Constant per-frame decay**
Every frame, whatever was in `heatmap_accum` is multiplied by the same factor (e.g. 0.95). That means you lose 5 % of your existing heat each frame, no matter where it is in the image.

– **Half-life calculation**
The number of frames it takes for a given “blob” of heat to drop to 50 % of its original value is

$$
  \text{half\_life (frames)}
    = \frac{\ln(0.5)}{\ln(\text{DECAY\_RATE})}.
$$

For `DECAY_RATE = 0.95`, that’s

$$
  \tfrac{\ln(0.5)}{\ln(0.95)} \approx 13.5 \text{ frames}.
$$

At 30 fps, that’s about 0.45 s before any given heat mark has halved.

---

### Tuning suggestions

* **Faster forgetting**: lower `DECAY_RATE` (e.g. 0.8 → half-life ≈ 3 frames).
* **Slower forgetting**: raise `DECAY_RATE` closer to 1 (e.g. 0.99 → half-life ≈ 69 frames).
* **Linear fade**: instead of multiplying, subtract a small constant each frame (but beware of negative values and loss of smooth exponential characteristic).




In this context, a **frame** is just one still image in your video’s sequence of images. Videos are nothing more than a rapid slideshow of frames—typically at 24, 30, or 60 frames per second (fps). Here’s how it maps to our code:

* **`frame_idx`**: an integer index (0, 1, 2, …) identifying each successive image in the capture.
* **Frame rate (fps)**: how many frames are shown (or captured) each second. At 30 fps, you get 30 frames every second—so each frame represents about 33 ms of real time.
* **Looping over frames**:

  ```python
  for frame_idx in range(n_frames):
      ret, frame = cap.read()
      …
  ```

  reads frame 0, frame 1, frame 2, … up to the last frame.

When we apply our decay every “frame,” we’re effectively saying “fade the heatmap by a fixed percentage each time the video advances by one image.” So with `DECAY_RATE = 0.95` at 30 fps, the heat from frame 0 has dropped to half its original intensity by around frame 13 (≈0.43 s later).


In [3]:
#whole video

import os
import glob
import cv2
import numpy as np
from pyneon import Recording

# ——— CONFIG ———
DATA_DIR       = '/Users/biancaschmid/Desktop/Spatial-Navigation-Group-1/Real_world_data'
OUTPUT_VIDEO   = '/Users/biancaschmid/Desktop/heatmap_overlay_dynamic.mp4'
GAUSSIAN_SIGMA = 50     # spread of each gaze point
CIRCLE_RADIUS  = 30     # radius (px) of each fixation dot
ALPHA_VIDEO    = 0.6    # weight for original frame
ALPHA_HEATMAP  = 0.4    # weight for heatmap overlay
DECAY_RATE     = 0.95   # multiply accumulator by this every frame (0<rate<1)
# ——————————

# 1) Load recording & get the gaze DataFrame
rec     = Recording(DATA_DIR)
stream  = rec.sync_gaze_to_video()
gaze_df = stream.data.dropna(subset=['gaze x [px]', 'gaze y [px]', 'frame_idx'])

# 2) Find and open the scene video
videos = sorted(glob.glob(os.path.join(DATA_DIR, '*.mp4')))
if not videos:
    raise FileNotFoundError(f"No .mp4 found in {DATA_DIR}")
cap      = cv2.VideoCapture(videos[0])
fps      = cap.get(cv2.CAP_PROP_FPS)
width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 3) Prepare writer & (now dynamic) accumulator
fourcc        = cv2.VideoWriter_fourcc(*'mp4v')
out           = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))
heatmap_accum = np.zeros((height, width), dtype=np.float32)

# 4) Frame‐by‐frame loop with decay
for frame_idx in range(n_frames):
    ret, frame = cap.read()
    if not ret:
        break

    # 4a) decay old heat
    heatmap_accum *= DECAY_RATE

    # 4b) add current gaze points
    pts = gaze_df[gaze_df['frame_idx'] == frame_idx]
    for _, row in pts.iterrows():
        x = int(row['gaze x [px]'])
        y = int(row['gaze y [px]'])
        if 0 <= x < width and 0 <= y < height:
            cv2.circle(heatmap_accum, (x, y), CIRCLE_RADIUS, color=1.0, thickness=-1)

    # 4c) blur & normalize
    heatmap_blur  = cv2.GaussianBlur(
        heatmap_accum, (0,0), GAUSSIAN_SIGMA, GAUSSIAN_SIGMA
    )
    heatmap_norm  = cv2.normalize(heatmap_blur, None, 0, 255, cv2.NORM_MINMAX)
    heatmap_uint8 = heatmap_norm.astype(np.uint8)

    # 4d) colorize & overlay
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    overlay       = cv2.addWeighted(frame, ALPHA_VIDEO, heatmap_color, ALPHA_HEATMAP, 0)

    # 4e) write out
    out.write(overlay)

# 5) clean up
cap.release()
out.release()
print(f"✅ Dynamic heatmap video saved to:\n   {OUTPUT_VIDEO}")


✅ Dynamic heatmap video saved to:
   /Users/biancaschmid/Desktop/heatmap_overlay_dynamic.mp4
